# 05 PPO：标准 Actor-Critic、GAE 与 PPO-Clip

本节自行实现标准在线 PPO 教学基线。Actor 生成回答，Critic 估计每个 Response Token 的状态价值，Old Policy 保存行为策略 Log Probability，确定性 Verifier 提供 Outcome Reward。PPO 与 GRPO 分开：PPO 使用 Critic 和 GAE，不依赖组内相对奖励。

DGX Spark 阶段只运行 CPU 数值 Smoke Test；不加载 Qwen3、不启动 vLLM、不进行 GPU Rollout 或更新。真实实验必须在 RTX 5090 官方训练/vLLM Profile 中同步串行执行。

## 学习目标

- 对齐 Prompt、Completion、EOS、Padding 和 Response Mask；
- 从 `r_t + gamma V_{t+1} - V_t` 推导 GAE 和 Return；
- 实现 Probability Ratio、PPO-Clip、Value Loss、Entropy 和 KL；
- 使用共享 Backbone + Value Head 的轻量 Actor-Critic 结构；
- 记录行为策略版本、Reward、Advantage、Clip Fraction 和 Explained Variance。

## 1. PPO 训练协议

同步串行流程为：`Actor Rollout → 保存 Token/Logprob/Mask → Verifier Reward → Critic Value → GAE → PPO Update → 保存 Adapter`。vLLM 和训练模型不同时驻留 GPU。Gravity 通常只作为能力保持评测，主 PPO 任务是 Equation Numeric 和 Bit Manipulation。

核心目标：

\[
r_t(\theta)=\exp(\log\pi_\theta-\log\pi_{old}),\quad
L_{clip}=-E[\min(r_t A_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)]
\]

\[
\delta_t=r_t+\gamma(1-d_t)V_{t+1}-V_t,\quad
A_t=\delta_t+\gamma\lambda(1-d_t)A_{t+1}
\]

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "ppo"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
SEED = 42
RUN_PPO = False
RUN_MODE = "smoke"
GROUP_TASKS = ("equation_numeric", "bit_manipulation")
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.7
TOP_P = 0.9
GAMMA = 0.99
GAE_LAMBDA = 0.95
PPO_CLIP_EPS = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01
KL_COEF = 0.0
LEARNING_RATE = 1e-5

np.random.seed(SEED)
torch.manual_seed(SEED)
print({"model_id": MODEL_ID, "run_ppo": RUN_PPO, "run_mode": RUN_MODE, "max_new_tokens": MAX_NEW_TOKENS, "tasks": GROUP_TASKS})

## 2. Rollout 数据结构与 Mask 约定

`input_ids` 包含 Prompt 和 Completion；`response_mask=1` 只覆盖 Completion Token，EOS 属于 Completion，Padding 为 0。`old_logprobs` 和 `old_values` 必须来自生成这批 Rollout 的行为策略版本，不能用更新后的 Policy 重新计算后冒充 Old Policy。

In [ ]:
@dataclass
class PPORollout:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    response_mask: torch.Tensor
    old_logprobs: torch.Tensor
    old_values: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
    policy_version: str
    verifier_version: str
    finish_reason: str
    truncated: bool
    task_type: str
    sample_id: str


def validate_rollout_alignment(rollout: PPORollout) -> None:
    shapes = [rollout.input_ids.shape, rollout.attention_mask.shape, rollout.response_mask.shape, rollout.old_logprobs.shape, rollout.old_values.shape, rollout.rewards.shape, rollout.dones.shape]
    if len({tuple(shape) for shape in shapes}) != 1:
        raise ValueError(f"rollout tensors are misaligned: {shapes}")
    if not torch.all((rollout.response_mask == 0) | (rollout.response_mask == 1)):
        raise ValueError("response_mask must be binary")
    if not torch.all(rollout.response_mask <= rollout.attention_mask):
        raise ValueError("response_mask cannot include padding")
    if rollout.truncated and torch.any(rollout.dones * rollout.response_mask):
        # A length-truncated sequence has no observed EOS; do not mark its last
        # token terminal merely because the tensor ended.
        raise ValueError("truncated rollout cannot claim an observed terminal EOS")


def outcome_reward(correct: bool, response_mask: torch.Tensor, task_type: str) -> torch.Tensor:
    """Binary outcome reward at the final observed response position."""
    reward = torch.zeros_like(response_mask, dtype=torch.float32)
    active = response_mask.bool()
    if active.any():
        final_position = torch.nonzero(active, as_tuple=False)[-1].item()
        reward[final_position] = 1.0 if correct else 0.0
    return reward


def build_cpu_rollout(batch_size: int = 2, time_steps: int = 7, vocab_size: int = 32) -> PPORollout:
    torch.manual_seed(SEED)
    input_ids = torch.randint(0, vocab_size, (batch_size, time_steps))
    attention_mask = torch.ones(batch_size, time_steps, dtype=torch.long)
    response_mask = torch.zeros(batch_size, time_steps, dtype=torch.long)
    response_mask[:, 2:] = 1
    old_logprobs = torch.randn(batch_size, time_steps) * 0.1
    old_values = torch.randn(batch_size, time_steps) * 0.1
    rewards = torch.zeros(batch_size, time_steps)
    rewards[:, -1] = torch.tensor([1.0, 0.0])
    dones = torch.zeros(batch_size, time_steps)
    dones[:, -1] = 1.0
    rollout = PPORollout(input_ids, attention_mask, response_mask, old_logprobs, old_values, rewards, dones, "cpu_fixture", "toy_verifier", "stop", False, "equation_numeric", "cpu-ppo")
    validate_rollout_alignment(rollout)
    return rollout

## 3. GAE、Return、PPO-Clip 和诊断指标

Padding 与终止状态都通过 Mask 处理。终止位置的 bootstrap 被关闭；Padding 不贡献 Advantage、Value Loss、Entropy 或 KL。`advantages` 只在有效 Response Token 上标准化。

In [ ]:
def masked_mean(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mask = mask.to(values.dtype)
    return (values * mask).sum() / mask.sum().clamp_min(eps)


def compute_gae(rewards: torch.Tensor, values: torch.Tensor, dones: torch.Tensor, response_mask: torch.Tensor, gamma: float = GAMMA, lam: float = GAE_LAMBDA) -> tuple[torch.Tensor, torch.Tensor]:
    if rewards.shape != values.shape or rewards.shape != dones.shape or rewards.shape != response_mask.shape:
        raise ValueError("rewards, values, dones and response_mask must share [B,T]")
    batch, time_steps = rewards.shape
    advantages = torch.zeros_like(rewards)
    next_advantage = torch.zeros(batch, device=rewards.device, dtype=rewards.dtype)
    for t in reversed(range(time_steps)):
        if t + 1 < time_steps:
            next_value = values[:, t + 1]
        else:
            next_value = torch.zeros(batch, device=values.device, dtype=values.dtype)
        nonterminal = 1.0 - dones[:, t]
        delta = rewards[:, t] + gamma * next_value * nonterminal - values[:, t]
        next_advantage = delta + gamma * lam * nonterminal * next_advantage
        advantages[:, t] = next_advantage
    advantages = advantages * response_mask.to(advantages.dtype)
    returns = (advantages + values) * response_mask.to(values.dtype)
    return advantages, returns


def normalize_masked(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mean = masked_mean(values, mask)
    variance = masked_mean((values - mean) ** 2, mask)
    return ((values - mean) / torch.sqrt(variance + eps)) * mask.to(values.dtype)


def ppo_clip_loss(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, advantages: torch.Tensor, response_mask: torch.Tensor, clip_eps: float = PPO_CLIP_EPS) -> dict[str, torch.Tensor]:
    ratio = torch.exp((new_logprobs - old_logprobs).clamp(-20, 20))
    unclipped = ratio * advantages
    clipped_ratio = ratio.clamp(1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * advantages
    objective = torch.minimum(unclipped, clipped)
    clip_fraction = ((ratio - clipped_ratio).abs() > 1e-8).to(torch.float32)
    return {
        "policy_loss": -masked_mean(objective, response_mask),
        "ratio": ratio.detach(),
        "clip_fraction": masked_mean(clip_fraction, response_mask),
    }


def value_loss(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean((values - returns.detach()) ** 2, response_mask)


def entropy_from_logits(logits: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    entropy = -(F.softmax(logits, dim=-1) * F.log_softmax(logits, dim=-1)).sum(dim=-1)
    return masked_mean(entropy, response_mask)


def approx_kl(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean(old_logprobs - new_logprobs, response_mask)


def explained_variance(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    mask = response_mask.bool()
    y, y_pred = returns.detach()[mask], values.detach()[mask]
    if y.numel() < 2 or torch.var(y) < 1e-8:
        return torch.tensor(0.0, device=values.device)
    return 1.0 - torch.var(y - y_pred) / torch.var(y)

## 4. 共享 Backbone + Scalar Value Head

真实 Qwen3 模型由 Transformers 加载，Value Head 读取最后一层 Hidden State；Policy Logits 仍来自 Causal LM Head。LoRA / QLoRA 的 Adapter 只更新 Actor Backbone 的可训练参数，Value Head 是否共享需通过显存和稳定性实验确认。这里的 Tiny 模型用于 CPU 机制测试。

In [ ]:
class TinyActorCritic(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.trunk = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh())
        self.policy_head = nn.Linear(hidden_size, vocab_size)
        self.value_head = nn.Linear(hidden_size, 1)
    def forward(self, input_ids, attention_mask=None):
        hidden = self.trunk(self.embedding(input_ids))
        return {"logits": self.policy_head(hidden), "values": self.value_head(hidden).squeeze(-1)}


class CausalLMValueHead(nn.Module):
    """GPU-side wrapper around a Transformers CausalLM."""
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        self.value_head = nn.Linear(hidden_size, 1, bias=False)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, use_cache=False)
        hidden = outputs.hidden_states[-1]
        return {"logits": outputs.logits, "values": self.value_head(hidden).squeeze(-1)}


def token_logprobs(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    return log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)


def align_response_mask(response_mask: torch.Tensor) -> torch.Tensor:
    # The token at position t is predicted by logits at position t-1.
    return response_mask[:, 1:].to(torch.float32)

## 5. 一次 PPO Update

Old Log Probability 和 Old Value 由 Rollout 固定；更新时只重新计算当前 Actor/Critic。总 Loss 为 Policy Loss + Value Loss − Entropy；本实现把 KL 记录为诊断，`KL_COEF=0` 时不重复惩罚。

In [ ]:
def ppo_update(model: nn.Module, optimizer: torch.optim.Optimizer, rollout: PPORollout, clip_eps: float = PPO_CLIP_EPS, value_coef: float = VALUE_COEF, entropy_coef: float = ENTROPY_COEF, kl_coef: float = KL_COEF) -> dict[str, float]:
    validate_rollout_alignment(rollout)
    outputs = model(rollout.input_ids, rollout.attention_mask)
    logits, values = outputs["logits"], outputs["values"]
    new_logprobs = token_logprobs(logits, rollout.input_ids)
    mask = align_response_mask(rollout.response_mask)
    old_logprobs = rollout.old_logprobs[:, 1:]
    old_values = rollout.old_values
    advantages, returns = compute_gae(rollout.rewards, old_values, rollout.dones, rollout.response_mask)
    advantages = normalize_masked(advantages, rollout.response_mask)
    policy = ppo_clip_loss(old_logprobs, new_logprobs, advantages[:, 1:], mask, clip_eps)
    value = value_loss(values, returns, rollout.response_mask)
    entropy = entropy_from_logits(logits[:, 1:, :], mask)
    kl = approx_kl(old_logprobs, new_logprobs, mask)
    total = policy["policy_loss"] + value_coef * value - entropy_coef * entropy + kl_coef * kl
    optimizer.zero_grad(set_to_none=True)
    total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return {
        "total_loss": float(total.detach()),
        "policy_loss": float(policy["policy_loss"].detach()),
        "value_loss": float(value.detach()),
        "entropy": float(entropy.detach()),
        "approx_kl": float(kl.detach()),
        "clip_fraction": float(policy["clip_fraction"].detach()),
        "advantage_mean": float(masked_mean(advantages, rollout.response_mask).detach()),
        "advantage_std": float(torch.sqrt(masked_mean(advantages ** 2, rollout.response_mask)).detach()),
        "explained_variance": float(explained_variance(values, returns, rollout.response_mask).detach()),
    }

## 6. GPU Rollout / PPO 入口（当前关闭）

真实入口使用 vLLM 生成并保留行为策略 Logprobs；如果 vLLM Profile 无法返回所需 Logprobs，必须停止并记录兼容性问题，不能用当前 Policy Logprobs 冒充 Old Policy。训练完成后保存 Adapter，再串行切回 vLLM。

In [ ]:
def run_vllm_ppo_rollout(prompt_rows: list[dict[str, object]], model_source: str, policy_version: str) -> list[dict[str, object]]:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    tokenizer = AutoTokenizer.from_pretrained(model_source, trust_remote_code=True, local_files_only=Path(model_source).exists())
    llm = LLM(model=model_source, tokenizer=model_source, tensor_parallel_size=1, gpu_memory_utilization=0.85, max_model_len=8192, enforce_eager=True)
    sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS, logprobs=1)
    rendered = [tokenizer.apply_chat_template([{"role": "user", "content": row["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for row in prompt_rows]
    outputs = llm.generate(rendered, sampling, use_tqdm=True)
    records = []
    for row, output in zip(prompt_rows, outputs):
        generated = output.outputs[0]
        if generated.logprobs is None:
            raise RuntimeError("behavior policy logprobs missing")
        records.append({
            "sample_id": row["id"], "task_type": row["task_type"], "prompt": row["prompt"],
            "completion": generated.text, "completion_token_ids": list(generated.token_ids or []),
            "prompt_token_ids": list(getattr(output, "prompt_token_ids", []) or []),
            "finish_reason": generated.finish_reason, "truncated": generated.finish_reason == "length",
            "policy_version": policy_version, "temperature": TEMPERATURE, "top_p": TOP_P,
            "behavior_logprobs": generated.logprobs, "source": "vllm_rollout",
        })
    del llm
    return records


def run_ppo_training(rollouts: list[PPORollout], sft_checkpoint: str) -> dict[str, object]:
    if not RUN_PPO:
        return {"status": "skipped_by_default", "reason": "RUN_PPO=False；DGX Spark 阶段不加载模型。"}
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(sft_checkpoint, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    model = CausalLMValueHead(base)
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
    logs = [ppo_update(model, optimizer, rollout) for rollout in rollouts]
    output_dir = RESULT_DIR / "ppo_smoke"
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_dir / "actor_critic_state.pt")
    return {"status": "completed", "updates": len(logs), "logs": logs, "checkpoint": str(output_dir)}


if RUN_PPO:
    raise RuntimeError("DGX Spark 阶段禁止 PPO GPU 执行")
print({"ppo_gpu": "disabled", "rollout": "disabled", "gravity_rl": "excluded_from_main_training"})

## 7. CPU Smoke Test

Smoke Test 使用 Tiny Actor-Critic 和人工构造的张量，只验证 GAE 终止状态、Mask、Ratio、Clip Fraction、Value/Entropy/KL 和反向传播。它不是 Qwen3 的 PPO 结果，也不证明真实 Reward 有效。

In [ ]:
rollout = build_cpu_rollout()
advantages, returns = compute_gae(rollout.rewards, rollout.old_values, rollout.dones, rollout.response_mask)
assert advantages.shape == rollout.rewards.shape
assert returns.shape == rollout.rewards.shape
assert torch.all(advantages[:, :2] == 0)
assert torch.all(returns[:, :2] == 0)

model = TinyActorCritic()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
logs = ppo_update(model, optimizer, rollout)
assert all(math.isfinite(value) for value in logs.values())
assert logs["clip_fraction"] >= 0.0
assert any(parameter.grad is not None for parameter in model.parameters())

# A terminal reward must not bootstrap through the next state.
terminal_rewards = torch.tensor([[0.0, 1.0]])
terminal_values = torch.tensor([[0.3, 0.4]])
terminal_dones = torch.tensor([[0.0, 1.0]])
terminal_mask = torch.ones_like(terminal_rewards)
terminal_adv, _ = compute_gae(terminal_rewards, terminal_values, terminal_dones, terminal_mask, gamma=0.9, lam=0.95)
expected_last = terminal_rewards[0, 1] - terminal_values[0, 1]
assert torch.isclose(terminal_adv[0, 1], expected_last)

ppo_cpu_report = {
    "status": "passed",
    "gae_shape_and_mask": True,
    "terminal_bootstrap_disabled": True,
    "ppo_loss_finite": True,
    "policy_gradient": True,
    "clip_fraction": logs["clip_fraction"],
    "approx_kl": logs["approx_kl"],
    "gpu_rollout": "not_run",
    "gpu_training": "not_run",
}
print(json.dumps(ppo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与限制

已完成标准 PPO 的核心数值实现和 CPU Smoke Test。当前没有真实 Qwen3 Actor、Critic、vLLM Rollout 或 GPU 更新结果。明天先用少量 Equation/Bit Rollout 检查行为 Logprobs、Response Mask、EOS 和 Critic Value 对齐，再运行一轮 PPO；若 Reward 上升但独立 Accuracy 不升，结论应记录为训练诊断而不是能力提升。